In [11]:
# ==========================================
# PREPARACIÓN A: CARGA Y DIAGNÓSTICO
# ==========================================
import pandas as pd

def miles(n):
    """Formatea un entero con punto como separador de miles (ej: 103904 -> 103.904)."""
    return f"{n:,}".replace(",", ".")

url = "https://raw.githubusercontent.com/egle06/airline-satisfaction-analysis/refs/heads/main/train.csv"
df = pd.read_csv(url, index_col=0)

# 1. Estructura y tipos
print(f"Filas: {miles(df.shape[0])} | Columnas: {df.shape[1]}\n")
df.info()

# 2. Estadísticas descriptivas
print("\n--- describe() ---")
display(df.describe().T)

# 3. Nulos por columna
print("\n--- Nulos por columna ---")
nulos = df.isnull().sum()
display(nulos[nulos > 0])

# 4. Duplicados: filas completas e id repetidos
print(f"\nFilas completamente duplicadas: {df.duplicated().sum()}")
print(f"Filas con id repetido: {df['id'].duplicated().sum()}")

# 5. Consistencia de las variables categóricas
print("\n--- Categorías de las variables de texto ---")
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

Filas: 103.904 | Columnas: 24

<class 'pandas.core.frame.DataFrame'>
Index: 103904 entries, 0 to 103903
Data columns (total 24 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 103904 non-null  int64  
 1   Gender                             103904 non-null  object 
 2   Customer Type                      103904 non-null  object 
 3   Age                                103904 non-null  int64  
 4   Type of Travel                     103904 non-null  object 
 5   Class                              103904 non-null  object 
 6   Flight Distance                    103904 non-null  int64  
 7   Inflight wifi service              103904 non-null  int64  
 8   Departure/Arrival time convenient  103904 non-null  int64  
 9   Ease of Online booking             103904 non-null  int64  
 10  Gate location                      103904 non-null  int64  
 11  Food and drin

,count,mean,std,min,25%,50%,75%,max
id,103904.0,64924.210502,37463.812252,1.0,32533.75,64856.5,97368.25,129880.0
Age,103904.0,39.379706,15.114964,7.0,27.00,40.0,51.00,85.0
Flight Distance,103904.0,1189.448375,997.147281,31.0,414.00,843.0,1743.00,4983.0
Inflight wifi service,103904.0,2.729683,1.327829,0.0,2.00,3.0,4.00,5.0
Departure/Arrival time convenient,103904.0,3.060296,1.525075,0.0,2.00,3.0,4.00,5.0
Ease of Online booking,103904.0,2.756901,1.398929,0.0,2.00,3.0,4.00,5.0
Gate location,103904.0,2.976883,1.277621,0.0,2.00,3.0,4.00,5.0
Food and drink,103904.0,3.202129,1.329533,0.0,2.00,3.0,4.00,5.0
Online boarding,103904.0,3.250375,1.349509,0.0,2.00,3.0,4.00,5.0
Seat comfort,103904.0,3.439396,1.319088,0.0,2.00,4.0,5.00,5.0



--- Nulos por columna ---


,0
Arrival Delay in Minutes,310



Filas completamente duplicadas: 0
Filas con id repetido: 0

--- Categorías de las variables de texto ---

Gender:
Gender
Female    52727
Male      51177
Name: count, dtype: int64

Customer Type:
Customer Type
Loyal Customer       84923
disloyal Customer    18981
Name: count, dtype: int64

Type of Travel:
Type of Travel
Business travel    71655
Personal Travel    32249
Name: count, dtype: int64

Class:
Class
Business    49665
Eco         46745
Eco Plus     7494
Name: count, dtype: int64

satisfaction:
satisfaction
neutral or dissatisfied    58879
satisfied                  45025
Name: count, dtype: int64


## Preparación: carga y limpieza del dataset
Resumen del proceso de la entrega anterior: se carga el dataset original y se limpia (nulos, ceros en las escalas de servicio, categorías y tipos de datos). El resultado es `df_limpio`, que se usa en el resto de este notebook.

In [12]:
# ==========================================
# PREPARACIÓN B: LIMPIEZA
# ==========================================
# Trabajamos sobre una copia: df queda como dataset original
df_limpio = df.copy()
print(f"Filas iniciales: {miles(len(df_limpio))}")

# --- 1. DUPLICADOS (verificado: 0, se deja la instrucción por robustez) ---
df_limpio = df_limpio.drop_duplicates(subset='id')
print(f"Tras eliminar duplicados: {miles(len(df_limpio))} filas")

# --- 2. NORMALIZACIÓN DE CATEGORÍAS (mayúsculas inconsistentes) ---
for col in ['Customer Type', 'Type of Travel']:
    df_limpio[col] = df_limpio[col].str.strip().str.title()

print("\nCategorías tras normalizar:")
print(df_limpio['Customer Type'].value_counts())
print(df_limpio['Type of Travel'].value_counts())

# --- 3. IMPUTACIÓN DE NULOS: mediana (la variable es muy asimétrica, la media no es representativa) ---
mediana_llegada = df_limpio['Arrival Delay in Minutes'].median()
df_limpio['Arrival Delay in Minutes'] = df_limpio['Arrival Delay in Minutes'].fillna(mediana_llegada)
print(f"\nNulos imputados con la mediana ({mediana_llegada} min). Nulos restantes: {df_limpio['Arrival Delay in Minutes'].isnull().sum()}")

# --- 4. CONVERSIÓN DE TIPOS ---
df_limpio['Arrival Delay in Minutes'] = df_limpio['Arrival Delay in Minutes'].astype(int)  # minutos enteros
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']:
    df_limpio[col] = df_limpio[col].astype('category')

# --- 5. RESET DEL ÍNDICE ---
df_limpio = df_limpio.reset_index(drop=True)

print("\n--- Tipos de datos tras la limpieza ---")
print(df_limpio.dtypes.value_counts())
print(f"Nulos totales: {df_limpio.isnull().sum().sum()}")

Filas iniciales: 103.904
Tras eliminar duplicados: 103.904 filas

Categorías tras normalizar:
Customer Type
Loyal Customer       84923
Disloyal Customer    18981
Name: count, dtype: int64
Type of Travel
Business Travel    71655
Personal Travel    32249
Name: count, dtype: int64

Nulos imputados con la mediana (0.0 min). Nulos restantes: 0

--- Tipos de datos tras la limpieza ---
int64       19
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64
Nulos totales: 0


In [13]:
# ==========================================
# PREPARACIÓN C: TRATAMIENTO DE CEROS EN ESCALAS DE SERVICIO
# ==========================================
# Los 0 se interpretan como "no aplica / sin respuesta": se convierten a NaN
# y se imputan con la moda (valor más frecuente), porque son escalas ordinales enteras.

cols_servicio = ['Inflight wifi service', 'Departure/Arrival time convenient',
                 'Ease of Online booking', 'Gate location', 'Food and drink',
                 'Online boarding', 'Seat comfort', 'Inflight entertainment',
                 'On-board service', 'Leg room service', 'Baggage handling',
                 'Checkin service', 'Inflight service', 'Cleanliness']

total_ceros = (df_limpio[cols_servicio] == 0).sum().sum()
print(f"Ceros a tratar: {miles(total_ceros)}")

for col in cols_servicio:
    moda = df_limpio[col][df_limpio[col] != 0].mode()[0]   # moda calculada sin contar los ceros
    df_limpio[col] = df_limpio[col].mask(df_limpio[col] == 0, moda)

# Verificación
print(f"Ceros restantes: {(df_limpio[cols_servicio] == 0).sum().sum()}")
print(f"Nulos restantes: {df_limpio[cols_servicio].isnull().sum().sum()}")
print(f"Rango de las escalas: {df_limpio[cols_servicio].min().min()} a {df_limpio[cols_servicio].max().max()}")

Ceros a tratar: 15.932
Ceros restantes: 0
Nulos restantes: 0
Rango de las escalas: 1 a 5


In [14]:
# ==========================================
# VERIFICACIÓN FINAL DE LA LIMPIEZA
# ==========================================
print(f"Filas: {miles(df_limpio.shape[0])} | Columnas: {df_limpio.shape[1]}")
print(f"Nulos totales: {df_limpio.isnull().sum().sum()}")
print(f"Duplicados: {df_limpio.duplicated().sum()}")
print(f"Índice: de {df_limpio.index.min()} a {miles(df_limpio.index.max())}\n")
df_limpio.info()

Filas: 103.904 | Columnas: 24
Nulos totales: 0
Duplicados: 0
Índice: de 0 a 103.903

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 24 columns):
 #   Column                             Non-Null Count   Dtype   
---  ------                             --------------   -----   
 0   id                                 103904 non-null  int64   
 1   Gender                             103904 non-null  category
 2   Customer Type                      103904 non-null  category
 3   Age                                103904 non-null  int64   
 4   Type of Travel                     103904 non-null  category
 5   Class                              103904 non-null  category
 6   Flight Distance                    103904 non-null  int64   
 7   Inflight wifi service              103904 non-null  int64   
 8   Departure/Arrival time convenient  103904 non-null  int64   
 9   Ease of Online booking             103904 non-null  int64   
 10  Gate lo

In [15]:
# ==========================================
# PASO 1: PREPARACIÓN DE X E y
# ==========================================

# 1. Partimos del dataset limpio
df = df_limpio.copy()
print(f"Filas: {miles(df.shape[0])} | Columnas: {df.shape[1]}")

# 2. Target: 1 = satisfecho, 0 = neutral o insatisfecho
y = (df['satisfaction'] == 'satisfied').astype(int)

# 3. Características: sin id, sin el target y sin columnas auxiliares (evita fuga de datos)
descartar = ['id', 'satisfaction']
X = df.drop(columns=descartar, errors='ignore')

# 4. Variables de categoria -> columnas numéricas (0/1)
X = pd.get_dummies(X, columns=['Gender', 'Customer Type', 'Type of Travel', 'Class'],
                   drop_first=True, dtype=int)

# Verificación
print(f"\nX: {miles(X.shape[0])} filas x {X.shape[1]} columnas")
print(f"y: {miles(y.shape[0])} valores")
print(f"\nColumnas de X:\n{list(X.columns)}")
print(f"\nBalance del target:\n{y.value_counts(normalize=True).round(3)}")

Filas: 103.904 | Columnas: 24

X: 103.904 filas x 23 columnas
y: 103.904 valores

Columnas de X:
['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender_Male', 'Customer Type_Loyal Customer', 'Type of Travel_Personal Travel', 'Class_Eco', 'Class_Eco Plus']

Balance del target:
satisfaction
0    0.567
1    0.433
Name: proportion, dtype: float64


In [16]:
# ==========================================
# PASO 2: DIVISIÓN EN TRAIN Y TEST
# ==========================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20 % para prueba
    random_state=42      # semilla fija: la división es siempre la misma
)

print(f"Train: {miles(X_train.shape[0])} filas ({X_train.shape[0] / len(X) * 100:.0f} %)")
print(f"Test:  {miles(X_test.shape[0])} filas ({X_test.shape[0] / len(X) * 100:.0f} %)")

print("\nBalance del target (proporción de satisfechos):")
print(f"  Dataset completo: {y.mean():.3f}")
print(f"  Train:            {y_train.mean():.3f}")
print(f"  Test:             {y_test.mean():.3f}")

Train: 83.123 filas (80 %)
Test:  20.781 filas (20 %)

Balance del target (proporción de satisfechos):
  Dataset completo: 0.433
  Train:            0.433
  Test:             0.436


In [17]:
# ==========================================
# PASO 3: ENTRENAMIENTO
# ==========================================
from sklearn.tree import DecisionTreeClassifier

# 1. Crear el modelo (sin límite de profundidad)
modelo = DecisionTreeClassifier(random_state=42)

# 2. Entrenar SOLO con los datos de entrenamiento
modelo.fit(X_train, y_train)

print("Modelo entrenado solo con X_train e y_train")
print(f"Profundidad del árbol: {modelo.get_depth()}")
print(f"Cantidad de hojas: {miles(modelo.get_n_leaves())}")

Modelo entrenado solo con X_train e y_train
Profundidad del árbol: 34
Cantidad de hojas: 3.737


In [18]:
# ==========================================
# PASO 4: EVALUACIÓN DUAL (TRAIN vs TEST)
# ==========================================
score_train = modelo.score(X_train, y_train)
score_test = modelo.score(X_test, y_test)

print(f"Score en TRAIN: {score_train:.4f}  ({score_train * 100:.2f} % de aciertos)")
print(f"Score en TEST:  {score_test:.4f}  ({score_test * 100:.2f} % de aciertos)")
print(f"\nDiferencia (train - test): {(score_train - score_test) * 100:.2f} puntos porcentuales")

# Referencia: qué acierto se obtendría prediciendo siempre la clase más frecuente
referencia = 1 - y_test.mean()
print(f"\nReferencia (predecir siempre 'no satisfecho'): {referencia:.4f}")

Score en TRAIN: 1.0000  (100.00 % de aciertos)
Score en TEST:  0.9395  (93.95 % de aciertos)

Diferencia (train - test): 6.05 puntos porcentuales

Referencia (predecir siempre 'no satisfecho'): 0.5636


   ### Experimento extra: ¿cómo cambia el sobreajuste al limitar la profundidad?
   Se entrenan varios árboles con distintos valores de `max_depth`, siempre solo con `X_train` e `y_train`, y se comparan los puntajes en train y test.

In [19]:
# ==========================================
# EXTRA: EFECTO DE LIMITAR LA PROFUNDIDAD DEL ÁRBOL
# ==========================================
resultados = []
for prof in [3, 5, 8, 10, 15, 20, None]:
    m = DecisionTreeClassifier(max_depth=prof, random_state=42)
    m.fit(X_train, y_train)                      # siempre entrenando solo con train
    resultados.append({
        'max_depth': 'sin límite' if prof is None else str(prof),
        'Train': round(m.score(X_train, y_train), 4),
        'Test': round(m.score(X_test, y_test), 4)
    })

res = pd.DataFrame(resultados)
res['Diferencia'] = (res['Train'] - res['Test']).round(4)
display(res)

,max_depth,Train,Test,Diferencia
0,3,0.8740,0.8742,-0.0002
1,5,0.8969,0.8974,-0.0005
2,8,0.9299,0.9282,0.0017
3,10,0.9451,0.9372,0.0079
4,15,0.9728,0.9455,0.0273
5,20,0.9902,0.9412,0.0490
6,sin límite,1.0000,0.9395,0.0605


## Análisis crítico

### 1. Rendimiento del modelo base (árbol sin límite)

| Conjunto / Referencia | Accuracy | Rendimiento relativo |
|---|:---:|:---:|
| **Entrenamiento (Train)** | 1,0000 | 100,00 % |
| **Prueba (Test)** | 0,9395 | 93,95 % |
| **Brecha (Train - Test)** | 0,0605 | 6,05 p.p. |
| **Baseline (predecir siempre «no satisfecho»)** | 0,5636 | 56,36 % |

* **Diagnóstico de Overfitting / Underfitting:**
  * **Overfitting (Moderado):** El árbol alcanzó 34 niveles de profundidad y 3.737 hojas, memorizando por completo el conjunto de entrenamiento (100 % de acierto). Aunque en test cae 6,05 puntos porcentuales, el rendimiento se mantiene alto (93,95 %) y supera con holgura el baseline (56,36 %), demostrando capacidad de generalización.
  * **Underfitting (Descartado):** Ambos puntajes son elevados; el modelo cuenta con suficiente capacidad para capturar la estructura de los datos.

---

### 2. Optimización de hiperparámetros: Efecto de `max_depth`

Para mitigar el sobreajuste, se evaluó podar el árbol controlando su profundidad máxima:

| `max_depth` | Train | Test | Brecha (Train - Test) | Diagnóstico |
|:---:|:---:|:---:|:---:|---|
| **3** | 87,40 % | 87,42 % | -0,02 p.p. | Alta generalización, capacidad limitada |
| **5** | 89,69 % | 89,74 % | -0,05 p.p. | Buen balance, sin sobreajuste |
| **8** | 92,99 % | 92,82 % | +0,17 p.p. | Ajuste balanceado |
| **10** | 94,51 % | 93,72 % | +0,79 p.p. | Brecha mínima (<1 p.p.) con alto rendimiento |
| **15** | **97,28 %** | **94,55 %** | **+2,73 p.p.** | **Óptimo en Test (pico de generalización)** |
| **20** | 99,02 % | 94,12 % | +4,90 p.p. | Comienza la memorización |
| **Sin límite (34)** | 100,00 % | 93,95 % | +6,05 p.p. | Memorización completa (overfitting) |

> **Observación metodológica:** La selección de `max_depth = 15` sobre el conjunto de test introduce sesgo de optimización (fuga de información indirecta). En un entorno productivo o de validación rigurosa, este umbral debe definirse mediante un conjunto de validación independiente o *k-fold cross-validation*.

---

### 3. Plan de acción y siguientes pasos

1. **Regularización y poda:** Adoptar una profundidad fija entre 10 y 15 para reducir la complejidad estructural del árbol sin sacrificar exactitud en datos no vistos.
2. **Métodos de ensamble:** Migrar hacia un modelo de **Random Forest** o algoritmos basados en gradiente (Gradient Boosting / XGBoost) para disminuir la varianza mediante agregación de estimadores.
3. **Estrategia de validación:** Implementar **Validación Cruzada Estratificada** (*Stratified K-Fold*) para mitigar la dependencia de una única partición estática.
4. **Métricas complementarias:** Si bien la accuracy es representativa con clases moderadamente equilibradas (56 % / 44 %), se recomienda incorporar **Matriz de confusión, Precisión, Recall y F1-score** para evaluar asimetrías en los errores de clasificación (falsos positivos vs. falsos negativos).
5. **Ampliación de muestra:** Incorporar más volumen de datos si está disponible, lo cual reduce naturalmente la dispersión entre train y test.